In [4]:
import os
import ctypes
import torch

os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = os.path.expanduser("~/.local/lib") + ":" + os.environ.get("LD_LIBRARY_PATH", "")

LIB = os.path.expanduser("~/.local/lib/libespeak-ng.so")
ctypes.CDLL(LIB)

os.environ["CUDA_VISIBLE_DEVICES"] = "7"
device = torch.device("cuda")

In [5]:
!espeak-ng --version

eSpeak NG text-to-speech: 1.52.0  Data at: /home/minh_duc/.local/share/espeak-ng-data


In [6]:
from neutts import NeuTTS
import soundfile as sf

tts = NeuTTS(
   backbone_repo="neuphonic/neutts-nano", # or 'neutts-nano-q4-gguf' with llama-cpp-python installed
   backbone_device="cuda",
   codec_repo="neuphonic/neucodec",
   codec_device="cuda",
   use_hier=True,
)
input_text = "Let me see: I'll give them a new pair of boots every Christmas.'"

ref_text = "samples/jo.txt"
ref_audio_path = "samples/jo.wav"

ref_text = open(ref_text, "r").read().strip()
ref_codes = tts.encode_reference(ref_audio_path)

wav, codes = tts.infer(input_text, ref_codes, ref_text, 
                       return_codes=True, sampling_scheme="orig")
sf.write("test.wav", wav, 24000)

Loading phonemizer...
Loading backbone from: neuphonic/neutts-nano on cuda ...


Loading codec from: neuphonic/neucodec on cuda ...


Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 14614.30it/s]
/home/minh_duc/miniconda3/envs/neutts_env/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


loaded PerthNet (Implicit) at step 250,000


In [7]:
print(tts.tokenizer.convert_tokens_to_ids("<|speech_0|>"))


128262


In [8]:
print(tts.tokenizer.convert_tokens_to_ids("<|SPEECH_GENERATION_END|>"))


128261


In [9]:
import random
import soundfile as sf
from IPython.display import Audio, display

# ---- CONFIG ----
data_dir = "/data2/minh_duc/from_hf/libritts/new.test.clean"
idx = 338  # hardcoded

# ---- LOAD KALDI FILES ----
wav_scp_path = os.path.join(data_dir, "wav.scp")
text_path = os.path.join(data_dir, "text")
utt2spk_path = os.path.join(data_dir, "utt2spk")

# read files
with open(wav_scp_path) as f:
    wav_lines = [l.strip().split(maxsplit=1) for l in f]

with open(text_path) as f:
    text_lines = [l.strip().split(maxsplit=1) for l in f]

with open(utt2spk_path) as f:
    utt2spk_lines = [l.strip().split() for l in f]

# convert to dict
wav_dict = dict(wav_lines)
text_dict = dict(text_lines)
utt2spk = dict(utt2spk_lines)

# ---- GET TARGET ----
utt_id = wav_lines[idx][0]
audio_path = wav_dict[utt_id]
target_text = text_dict[utt_id]
speaker = utt2spk[utt_id]

print(f"Target utt_id: {utt_id}")
print(f"Speaker: {speaker}")
print(f"Text: {target_text}")

# ---- PLAY TARGET AUDIO ----
audio, sr = sf.read(audio_path)
display(Audio(audio, rate=sr))

# ---- FIND REFERENCE FROM SAME SPEAKER ----
same_spk_utts = [u for u, s in utt2spk.items() if s == speaker and u != utt_id]
ref_utt = random.choice(same_spk_utts)

ref_audio_path = wav_dict[ref_utt]
ref_text = text_dict[ref_utt]

print("\nReference utt_id:", ref_utt)
print("Reference text:", ref_text)

# ---- PLAY REF AUDIO ----
ref_audio, sr = sf.read(ref_audio_path)
display(Audio(ref_audio, rate=sr))

Target utt_id: 2300_131720_000041_000001
Speaker: 2300
Text: Note has already been made of the first Edison plants afloat on the Jeannette and Columbia, and the first commercial plant in the New York lithographic establishment.



Reference utt_id: 2300_131720_000033_000005
Reference text: Some idea of the confidence inspired by the fame of Edison at this period is shown by the fact that the first theatre ever lighted from a central station by incandescent lamps was designed this year, and opened in eighteen eighty four at Brockton with an equipment of three hundred lamps.


In [10]:
# encode reference
ref_codes = tts.encode_reference(ref_audio_path)

# inference
wav, codes = tts.infer(
    target_text,
    ref_codes,
    ref_text,
    return_codes=True,
    sampling_scheme="eas"
)

# save + play
sf.write("/home/minh_duc/neutts/eas.wav", wav, 24000)
display(Audio(wav, rate=24000))

In [11]:
wav, codes = tts.infer(
    target_text,
    ref_codes,
    ref_text,
    return_codes=True,
    sampling_scheme="ras_k50_win25"
)

sf.write("/home/minh_duc/neutts/ras_k50_win25.wav", wav, 24000)
display(Audio(wav, rate=24000))

In [12]:
wav, codes = tts.infer(
    target_text,
    ref_codes,
    ref_text,
    return_codes=True,
    sampling_scheme="rank_eas_hier"
)

sf.write("/home/minh_duc/neutts/rank_eas_hier.wav", wav, 24000)
display(Audio(wav, rate=24000))

In [13]:
wav, codes = tts.infer(
    target_text,
    ref_codes,
    ref_text,
    return_codes=True,
    sampling_scheme="rank_ras_hier"
)

sf.write("/home/minh_duc/neutts/rank_ras_hier.wav", wav, 24000)
display(Audio(wav, rate=24000))